# Car Classification - 200k Dataset Training on Kaggle

This notebook is configured to run the `train_v2_augmented.py` script on Kaggle to generate a dataset of roughly 200,000 images (28,572 per class) and train the YOLOv8 classification model.

## 🚀 Instructions for Kaggle:
1. **Enable GPU**: On the right panel, go to **Notebook Options > Accelerator** and select **GPU T4 x2** or **GPU P100**.
2. **Enable Internet**: Ensure **Internet** is toggled **ON** in the Notebook Options so HuggingFace datasets and YOLOv8 weights can download.
3. **Upload Local Data (Optional)**: If you possess local datasets such as `Datasets` or `Car-Bike-Dataset`, upload them to Kaggle as a new Dataset, then replace `car-classification-data` in cell 3 with your dataset name.
4. Click **Run All**.

In [2]:
# 1. Install necessary dependencies
!pip install ultralytics albumentations datasets seaborn scikit-learn numpy opencv-python pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.6 MB/s eta 0:00:00a 0:00:01


In [3]:
import os, shutil

# 2. Prepare workspace: Copy local datasets if you uploaded them to Kaggle Inputs
INPUT_DATASET_NAME = "car-classification-data"  # <-- Change this if your Kaggle dataset name is different

INPUT_DIR = f"/kaggle/input/datasets/polanaeem/cardata"
WORKING_DIR = "/kaggle/working"

if os.path.exists(INPUT_DIR):
    # The zip wraps everything under 'car_data/' -- check both locations
    car_data_subdir = os.path.join(INPUT_DIR, 'car_data')
    search_dir = car_data_subdir if os.path.exists(car_data_subdir) else INPUT_DIR
    print(f"Looking for datasets in: {search_dir}")
    for folder_name in ['Datasets', 'Car-Bike-Dataset']:
        src = os.path.join(search_dir, folder_name)
        dst = os.path.join(WORKING_DIR, folder_name)
        if os.path.exists(src) and not os.path.exists(dst):
            shutil.copytree(src, dst)
            print(f"Copied {folder_name} to working directory.")
        elif not os.path.exists(src):
            print(f"Warning: {folder_name} not found in {search_dir}")
else:
    print("No custom input dataset found. Will download from HuggingFace.")

print("Workspace ready!")

Looking for datasets in: /kaggle/input/datasets/polanaeem/cardata/car_data
Copied Datasets to working directory.
Copied Car-Bike-Dataset to working directory.
Workspace ready!


In [4]:
# 3. Imports & Configuration
import os, shutil, random, sys
import numpy as np
import cv2
from PIL import Image
from ultralytics import YOLO
from sklearn.metrics import classification_report
import seaborn as sns
import matplotlib.pyplot as plt
import albumentations as A

DATA_DIR    = "data"
TRAIN_DIR   = os.path.join(DATA_DIR, "train")
VAL_DIR     = os.path.join(DATA_DIR, "val")
TEST_DIR    = os.path.join(DATA_DIR, "test")
CLASSES     = ['compact', 'motorcycle', 'sedan', 'suv', 'truck', 'van']
PROJECT_NAME = 'car_classification_project'
MODEL_NAME   = 'augmented_model'
TARGET_COUNT = 28572   # 7 classes * 28572 = ~200k images
EPOCHS       = 30

def get_augmentation_pipeline():
    return A.Compose([
        A.RandomBrightnessContrast(p=0.5, brightness_limit=0.2, contrast_limit=0.2),
        A.MotionBlur(blur_limit=5, p=0.3),
        A.GaussNoise(std_range=(0.05, 0.2), p=0.3),
        A.RandomShadow(num_shadows_limit=(1, 3),
                       shadow_dimension=5, shadow_roi=(0, 0.5, 1, 1), p=0.4),
        A.Rotate(limit=10, p=0.5, border_mode=cv2.BORDER_CONSTANT, fill=0),
    ])

def balance_and_augment_dataset(directory, target_count=None):
    print(f"\n--- Checking Class Balance in {directory} ---")
    counts = {}
    for cls in CLASSES:
        cls_path = os.path.join(directory, cls)
        os.makedirs(cls_path, exist_ok=True)
        counts[cls] = len([f for f in os.listdir(cls_path)
                           if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    print(f"Current distribution: {counts}")
    if sum(counts.values()) == 0:
        print("No data found! Skipping."); return
    final_target = max(max(counts.values()), target_count or 0)
    print(f"Target count per class: ~{final_target}")
    aug = get_augmentation_pipeline()
    for cls in CLASSES:
        current = counts[cls]
        if current == 0:
            print(f"WARNING: {cls} has 0 images!"); continue
        if current < final_target:
            diff  = final_target - current
            cls_path = os.path.join(directory, cls)
            files = [f for f in os.listdir(cls_path)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            to_aug = random.choices(files, k=diff)
            print(f"  Augmenting {cls}: +{diff} images...")
            for i, fname in enumerate(to_aug):
                dst = os.path.join(cls_path, f"aug_{i}_{fname}")
                if os.path.exists(dst): continue
                try:
                    img = cv2.imread(os.path.join(cls_path, fname))
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    out = aug(image=img)['image']
                    cv2.imwrite(dst, cv2.cvtColor(out, cv2.COLOR_RGB2BGR))
                except Exception as e:
                    print(f"    Error: {e}")
            print(f"    Done.")

def load_datasets_if_needed():
    print("Downloading datasets...")
    from datasets import load_dataset, DownloadConfig
    for split in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
        for cls in CLASSES:
            os.makedirs(os.path.join(split, cls), exist_ok=True)

    def get_split_dir():
        r = random.random()
        if r < 0.8: return TRAIN_DIR
        elif r < 0.9: return VAL_DIR
        else: return TEST_DIR

    def map_stanford(label):
        n = label.lower()
        if 'sedan' in n or 'coupe' in n or 'wagon' in n: return 'sedan'
        if 'suv'   in n: return 'suv'
        if 'truck' in n or 'pickup' in n: return 'truck'
        if 'van'   in n or 'minivan' in n: return 'van'
        if 'hatchback' in n: return 'compact'
        return None

    def map_extra(label):
        n = label.lower()
        if 'motorcycle' in n or 'bike' in n: return 'motorcycle'
        if 'truck' in n or 'pickup' in n: return 'truck'
        return None

    # Stanford Cars
    try:
        print("Loading Stanford Cars...")
        ds = load_dataset("tanganke/stanford_cars", split="train+test")
        cnt = 0
        for i, item in enumerate(ds):
            cls = map_stanford(ds.features['label'].int2str(item['label']))
            if cls:
                d = get_split_dir()
                try: item['image'].convert('RGB').save(os.path.join(d, cls, f"stanford_{i}.jpg")); cnt+=1
                except: pass
        print(f"Stanford Cars: {cnt} images.")
    except Exception as e:
        print(f"Stanford Cars error: {e}")

    # Local datasets (already copied to /kaggle/working in Cell 2)
    local_sources = [
        {"root": "Datasets/Datasets", "mapping": {"truck": "truck"}},
        {"root": "Datasets",          "mapping": {"truck": "truck"}},
        {"root": "Car-Bike-Dataset",  "mapping": {"Bike": "motorcycle"}},
    ]
    local_cnt = 0
    for src in local_sources:
        if not os.path.exists(src['root']):
            print(f"Skipping {src['root']} (not found)"); continue
        for lbl, cls in src['mapping'].items():
            sp = os.path.join(src['root'], lbl)
            if not os.path.exists(sp): continue
            for fname in os.listdir(sp):
                if not fname.lower().endswith(('.jpg','.jpeg','.png','.bmp','.webp')): continue
                d = get_split_dir()
                dst = os.path.join(d, cls, f"local_{lbl}_{fname}")
                if os.path.exists(dst): continue
                try:
                    with Image.open(os.path.join(sp, fname)) as img:
                        img.convert('RGB').save(dst)
                    local_cnt += 1
                except: pass
    print(f"Local images processed: {local_cnt}")

print("All functions defined. Ready to run.")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
All functions defined. Ready to run.


In [5]:
# 4. Download datasets (skipped if data already exists)
load_datasets_if_needed()

Loading Stanford Cars...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/504M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/485M [00:00<?, ?B/s]

data/test-00000-of-00002.parquet:   0%|          | 0.00/513M [00:00<?, ?B/s]

...

Generating pixelate split:   0%|          | 0/8041 [00:00<?, ? examples/s]

Generating spatter split:   0%|          | 0/8041 [00:00<?, ? examples/s]

Stanford Cars: 11812 images.


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Local images processed: 2528


In [6]:
# 5. Augment & balance train set to ~200k (7 classes * 28,572)
balance_and_augment_dataset(TRAIN_DIR, target_count=17000)


--- Checking Class Balance in data/train ---
Current distribution: {'compact': 894, 'motorcycle': 1624, 'sedan': 5279, 'suv': 2273, 'truck': 517, 'van': 868}
Target count per class: ~17000
  Augmenting compact: +16106 images...
    Done.
  Augmenting motorcycle: +15376 images...
    Done.
  Augmenting sedan: +11721 images...
    Done.
  Augmenting suv: +14727 images...
    Done.
  Augmenting truck: +16483 images...
    Done.
  Augmenting van: +16132 images...
    Done.


In [7]:
balance_and_augment_dataset(VAL_DIR, target_count=3000)


--- Checking Class Balance in data/val ---
Current distribution: {'compact': 101, 'motorcycle': 200, 'sedan': 663, 'suv': 275, 'truck': 85, 'van': 92}
Target count per class: ~3000
  Augmenting compact: +2899 images...
    Done.
  Augmenting motorcycle: +2800 images...
    Done.
  Augmenting sedan: +2337 images...
    Done.
  Augmenting suv: +2725 images...
    Done.
  Augmenting truck: +2915 images...
    Done.
  Augmenting van: +2908 images...
    Done.


In [8]:
balance_and_augment_dataset(TEST_DIR, target_count=3000)


--- Checking Class Balance in data/test ---
Current distribution: {'compact': 108, 'motorcycle': 176, 'sedan': 648, 'suv': 307, 'truck': 99, 'van': 131}
Target count per class: ~3000
  Augmenting compact: +2892 images...
    Done.
  Augmenting motorcycle: +2824 images...
    Done.
  Augmenting sedan: +2352 images...
    Done.
  Augmenting suv: +2693 images...
    Done.
  Augmenting truck: +2901 images...
    Done.
  Augmenting van: +2869 images...
    Done.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from ultralytics import YOLO
import pandas as pd
import copy
import time
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# Global cache for dataloaders
_DATALOADERS_CACHE = None

def get_dataloaders(data_dir, imgsz, batch_size):
    global _DATALOADERS_CACHE
    if _DATALOADERS_CACHE is not None:
        return _DATALOADERS_CACHE

    data_transforms = {
        'train': transforms.Compose([
            transforms.Resize((imgsz, imgsz)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ]),
        'val': transforms.Compose([
            transforms.Resize((imgsz, imgsz)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ]),
        'test': transforms.Compose([
            transforms.Resize((imgsz, imgsz)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ]),
    }

    image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x])
                      for x in ['train', 'val', 'test']}
    
    dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=batch_size,
                                                 shuffle=(x == 'train'), num_workers=4)
                  for x in ['train', 'val', 'test']}
    
    dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val', 'test']}
    class_names = image_datasets['train'].classes

    _DATALOADERS_CACHE = (dataloaders, dataset_sizes, class_names)
    return _DATALOADERS_CACHE

def plot_confusion_matrix(y_true, y_pred, class_names, model_name):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(f'Confusion Matrix: {model_name}')
    plt.tight_layout()
    plt.savefig(f'cm_{model_name}.png')
    plt.show()
    plt.close()

In [ ]:
def build_pytorch_model(model_name, num_classes):
    if model_name == 'resnet18':
        model = models.resnet18(weights='DEFAULT')
        num_ftrs = model.fc.in_features
        model.fc = nn.Linear(num_ftrs, num_classes)
    elif model_name == 'efficientnet_b0':
        model = models.efficientnet_b0(weights='DEFAULT')
        num_ftrs = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(num_ftrs, num_classes)
    elif model_name == 'mobilenet_v3':
        model = models.mobilenet_v3_large(weights='DEFAULT')
        num_ftrs = model.classifier[3].in_features
        model.classifier[3] = nn.Linear(num_ftrs, num_classes)
    else:
        raise ValueError(f"Unknown model name: {model_name}")
    return model

In [ ]:
def train_model(model_name, data_dir, epochs, batch_size, imgsz, patience, device, explicit_weight_path=None):
    print(f"\n{'='*50}\nTraining Model: {model_name}\n{'='*50}")
    start_time = time.time()
    
    found_weight = None
    if explicit_weight_path and os.path.exists(explicit_weight_path):
        found_weight = explicit_weight_path
    
    # Handle both yolo and torch typical extensions
    names_to_check = [f'best_{model_name}.pth', f'best_{model_name}.pt', f'{model_name}.pth']
    if model_name == 'yolov8': names_to_check.append('best_yolov8.pt')
    
    # Check current directory first if not explicitly set
    if not found_weight:
        for n in names_to_check:
            if os.path.exists(n):
                found_weight = n
                break
            
        # Alternatively, scan Kaggle datasets standard input directory if not found locally
        if not found_weight and os.path.exists('/kaggle/input'):
            for root, _, files in os.walk('/kaggle/input'):
                for f in files:
                    if f in names_to_check:
                        found_weight = os.path.join(root, f)
                        break
                if found_weight:
                    break
    
    elif explicit_weight_path and not os.path.exists(explicit_weight_path):
        print(f"[WARNING] Explicit weight path {explicit_weight_path} not found! Defaulting to train.")

    if model_name == 'yolov8':
        if found_weight:
            print(f"⭐ Loading pre-trained weights from {found_weight}. Skipping training!")
            import shutil
            if found_weight != 'best_yolov8.pt':
                shutil.copy(found_weight, 'best_yolov8.pt')
            return YOLO('best_yolov8.pt'), 0.0
            
        model = YOLO('yolov8n-cls.pt')
        yolo_device = 0 if device.type != 'cpu' else 'cpu'
        
        model.train(
            data=data_dir,
            epochs=epochs,
            imgsz=imgsz,
            batch=batch_size,
            workers=4,
            patience=patience,
            optimizer='AdamW',
            lr0=0.001,
            project='benchmark_runs',
            name='yolov8_training',
            device=yolo_device
        )
        
        best_yolo_path = os.path.join('benchmark_runs', 'yolov8_training', 'weights', 'best.pt')
        if os.path.exists(best_yolo_path):
            import shutil
            shutil.copy(best_yolo_path, 'best_yolov8.pt')
            trained_model = YOLO('best_yolov8.pt')
        else:
            trained_model = model
            
        train_time = time.time() - start_time
        return trained_model, train_time
        
    else:
        dataloaders, dataset_sizes, class_names = get_dataloaders(data_dir, imgsz, batch_size)
        num_classes = len(class_names)
        model = build_pytorch_model(model_name, num_classes).to(device)
        
        if found_weight:
            print(f"⭐ Loading pre-trained weights from {found_weight}. Skipping training!")
            model.load_state_dict(torch.load(found_weight, map_location=device))
            import shutil
            local_dst = f'best_{model_name}.pth'
            if found_weight != local_dst:
                shutil.copy(found_weight, local_dst)
            return model, 0.0
        
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)
        
        best_model_wts = copy.deepcopy(model.state_dict())
        best_acc = 0.0
        epochs_no_improve = 0

        for epoch in range(epochs):
            print(f'Epoch {epoch+1}/{epochs}')
            print('-' * 10)

            for phase in ['train', 'val']:
                if phase == 'train':
                    model.train()
                else:
                    model.eval()

                running_loss = 0.0
                running_corrects = 0

                for inputs, labels in dataloaders[phase]:
                    inputs = inputs.to(device)
                    labels = labels.to(device)
                    optimizer.zero_grad()

                    with torch.set_grad_enabled(phase == 'train'):
                        outputs = model(inputs)
                        _, preds = torch.max(outputs, 1)
                        loss = criterion(outputs, labels)

                        if phase == 'train':
                            loss.backward()
                            optimizer.step()

                    running_loss += loss.item() * inputs.size(0)
                    running_corrects += torch.sum(preds == labels.data)
                
                if phase == 'train' and scheduler is not None:
                    scheduler.step()

                epoch_loss = running_loss / dataset_sizes[phase]
                epoch_acc = running_corrects.double() / dataset_sizes[phase]
                print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

                if phase == 'val':
                    if epoch_acc > best_acc:
                        best_acc = epoch_acc
                        best_model_wts = copy.deepcopy(model.state_dict())
                        epochs_no_improve = 0
                    else:
                        epochs_no_improve += 1

            if epochs_no_improve >= patience:
                print(f"Early stopping triggered after {epoch+1} epochs.")
                break
            print()

        train_time = time.time() - start_time
        print(f'Training complete in {train_time // 60:.0f}m {train_time % 60:.0f}s')
        print(f'Best val Acc: {best_acc:4f}')

        model.load_state_dict(best_model_wts)
        torch.save(model.state_dict(), f'best_{model_name}.pth')
        
        return model, train_time

def evaluate_model(model, model_name, data_dir, imgsz, batch_size, device):
    print(f"\nEvaluating {model_name} on test set...")
    
    dataloaders, dataset_sizes, class_names = get_dataloaders(data_dir, imgsz, batch_size)
    test_loader = dataloaders['test']
    
    y_true = []
    y_pred = []
    top1_correct = 0
    top5_correct = 0
    total = 0

    if model_name == 'yolov8':
        test_dir = os.path.join(data_dir, 'test')
        for class_idx, class_name in enumerate(class_names):
            cls_path = os.path.join(test_dir, class_name)
            if not os.path.exists(cls_path):
                continue
            
            for file in os.listdir(cls_path):
                img_path = os.path.join(cls_path, file)
                try:
                    results = model(img_path, verbose=False)
                    probs = results[0].probs
                    pred_class_idx = probs.top1
                    top5_class_indices = probs.top5
                    if isinstance(top5_class_indices, torch.Tensor):
                        top5_class_indices = top5_class_indices.tolist()
                    
                    y_true.append(class_idx)
                    y_pred.append(pred_class_idx)
                    
                    if pred_class_idx == class_idx:
                        top1_correct += 1
                    if class_idx in top5_class_indices:
                        top5_correct += 1
                    total += 1
                except Exception as e:
                    pass
    else:
        model.eval()
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)
                outputs = model(inputs)
                
                _, preds = torch.max(outputs, 1)
                
                y_true.extend(labels.cpu().numpy())
                y_pred.extend(preds.cpu().numpy())
                
                top1_correct += torch.sum(preds == labels.data).item()
                
                k = min(5, len(class_names))
                _, topk_preds = outputs.topk(k, 1, True, True)
                topk_preds = topk_preds.t()
                correct_topk = topk_preds.eq(labels.view(1, -1).expand_as(topk_preds))
                top5_correct += correct_topk.reshape(-1).float().sum(0).item()
                
                total += labels.size(0)

    top1_acc = top1_correct / total if total > 0 else 0
    top5_acc = top5_correct / total if total > 0 else 0
    
    print(f"\n{model_name} Test Top-1 Acc: {top1_acc:.4f}")
    print(f"{model_name} Test Top-5 Acc: {top5_acc:.4f}")
    
    if total > 0:
        print("\nClassification Report:")
        print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))
        plot_confusion_matrix(y_true, y_pred, class_names, model_name)
        
    return top1_acc, top5_acc


In [ ]:
import ipywidgets as widgets
from IPython.display import display

print("==========================================================")
print("📥 INTERACTIVE LOCAL FILE UPLOAD")
print("==========================================================")
print("> Click the 'Upload' button below to select models like 'best_resnet18.pth' directly from your PC.")

# FileUpload widget allowing .pth or .pt files, and multiple at once
uploader = widgets.FileUpload(accept='.pth, .pt', multiple=True)
display(uploader)


In [ ]:
# MUST run this cell AFTER uploading files in the cell above.
saved_files = []

if uploader.value:
    print("Found files to extract, unpacking them into working directory...")
    for item in uploader.value:
        # Handle ipywidgets standard change between v7.x and v8.0
        if isinstance(item, dict):
            # ipywidgets >= 8 syntax
            name = item['name']
            content = item['content']
        else:
            # older widget syntax
            name = item
            content = uploader.value[item]['content']
            
        with open(name, 'wb') as f:
            f.write(content)
        saved_files.append(name)
        print(f"✅ Successfully processed: {name}")
else:
    print("⚠️ No files attached yet. If you wanted to upload, make sure you selected the files first.")


In [ ]:
DATA_DIR = "data"
IMG_SIZE = 224
BATCH_SIZE = 128
EPOCHS = 30
PATIENCE = 10

# ==================================================================
# ==================== PRE-TRAINED WEIGHTS =========================
# IF YOU UPLOADED YOUR TRAINED MODELS DIRECTLY TO THE NOTEBOOK,
# MAP THE MODEL NAME TO THE SPECIFIC FILE PATH BELOW TO SKIP TRAINING.
PRETRAINED_WEIGHTS = {
    'resnet18': 'best_resnet18.pth',                   # Default to the typical name
    'efficientnet_b0': 'best_efficientnet_b0.pth',     # Default to the typical name
    'mobilenet_v3': None,                              # Keep None to train it normally
    'yolov8': None                                     # Keep None to train it normally
}
# ==================================================================

MODELS_TO_BENCHMARK = ['resnet18', 'efficientnet_b0', 'mobilenet_v3', 'yolov8']
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print("="*50)
print("🚗 CAR CLASSIFICATION BENCHMARKING FRAMEWORK 🚗")
print("="*50)

results_records = []

for model_name in MODELS_TO_BENCHMARK:
    trained_model, train_time = train_model(
        model_name, DATA_DIR, EPOCHS, BATCH_SIZE, IMG_SIZE, PATIENCE, DEVICE, PRETRAINED_WEIGHTS.get(model_name)
    )
    top1, top5 = evaluate_model(
        trained_model, model_name, DATA_DIR, IMG_SIZE, BATCH_SIZE, DEVICE
    )
    results_records.append({
        'Model': model_name,
        'Top1': top1,
        'Top5': top5,
        'Training Time (s)': train_time
    })
    
    del trained_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\n{'='*50}\nBENCHMARK RESULTS\n{'='*50}")
df = pd.DataFrame(results_records)
df['Training Time Formatted'] = df['Training Time (s)'].apply(lambda x: f"{int(x // 60)}m {int(x % 60)}s")
df['Top1'] = df['Top1'].round(4)
df['Top5'] = df['Top5'].round(4)

display_df = df[['Model', 'Top1', 'Top5', 'Training Time Formatted']]
print(display_df.to_string(index=False))

df.to_csv('benchmark_results.csv', index=False)
print("\nResults saved to benchmark_results.csv")
